# AI Jobbmatchning

Notebook-version av programmet. Koden är indelad i logiska celler för att göra projektet enklare att läsa, testa och förklara.

# 1. Importer
Innehåller mindre funktioner som används
på flera olika ställen i programmet.

In [ ]:
#========================================
#       IMPORTER
#========================================

import requests              # Importerar requests för HTTP-anrop till API:t.
import json                  # Importerar json för att läsa och spara JSON-data.
from datetime import datetime  # Importerar datetime för datum och tid.


# 2. Hjälpmetoder
Innehåller basklasser och underklasser
som hanterar data och information

In [ ]:
#========================================
#       HJÄLPMETODER
#========================================

def normalisera(text):
    # Normaliserar text genom att göra den gemener och ta bort onödiga mellanslag.
    return text.lower().strip()


def visa_kompetenser(kompetenser):
    # Omvandlar en lista med kompetenser till en läsbar text.
    return ", ".join(kompetenser) if kompetenser else "Inga"

# 3. Basklass och profil
Innehåller basklasser och underklasser
som hanterar data och information

In [ ]:
#========================================
#       DATAKLASSER
#========================================

class MatchningsData:
    # Basklass som innehåller gemensam data för olika typer av profiler.
    def __init__(self):
        self.plats = []
        self.kompetenser = []


class Profil(MatchningsData):
    # Profil ärver gemensam data från MatchningsData och lägger till jobb.
    def __init__(self):
        super().__init__()
        self.jobb = []

    def input_jobb(self):
        # Hämtar användarens önskade jobb och sparar dem i profilen.
        self._input("Vad hade du velat jobba som? ", "jobb")

    def input_plats(self):
        # Hämtar användarens önskade platser och sparar dem i profilen.
        self._input("Vart hade du velat jobba? ", "plats")

    def input_kompetenser(self):
        # Hämtar användarens kompetenser och sparar dem i profilen.
        self._input("Vad har du för kompetenser? ", "kompetenser")

    def _input(self, text, attribut):
        # Hanterar användarinmatning och sparar normaliserade värden i rätt lista.
        print("\n" + "=" * 40)

        try:
            värden = [ #list comprehension. --> "Skapa en lista med normaliserade x, för varje x som användaren skrev, så länge x inte är tom."
                normalisera(x)
                for x in input(text).split(",")
                if x.strip()
            ]
            getattr(self, attribut).extend(värden) #gåtill objektet (self) och hämta attributet (plats, kompetenser eller jobb) och lägg till värdena i listan.

        except (KeyboardInterrupt, EOFError):
            print("\nInmatningen avbröts.")               # Informerar användaren att inmatningen avbröts.


## 4. Jobbannons och matchare
Innehåller klasser som utför beräkningar,
matchningar och hanterar programmets logik.

In [ ]:
class Jobbannons(MatchningsData):
    # Representerar en jobbannons med information om jobb, plats, kompetenser och länk.
    def __init__(self, jobb, plats, kompetenser, länk):
        super().__init__()
        self.jobb = jobb
        self.plats = plats
        self.kompetenser = kompetenser
        self.länk = länk


class Matchare:
    # Ansvarar för att jämföra en profil med jobbannonser och beräkna matchningspoäng.
    def __init__(self, profil, arbetsmarknad):
        self.profil = profil
        self.arbetsmarknad = arbetsmarknad

    def match(self):
        # Går igenom jobbannonser, beräknar matchningspoäng och sorterar resultaten.
        matchningar = []

        for jobb in self.arbetsmarknad:
            jobb_matchar = False #utgår från att jobbet inte matchar, tills vi hittar ett jobb som matchar.

            for önskat_jobb in self.profil.jobb: #går igenom vad användaren vill jobba med, och kollar om det finns i jobbannonsen.
                if önskat_jobb in jobb.jobb: #om det stämmer
                    jobb_matchar = True 
                    break #avslutas loopen

            if not jobb_matchar:
                continue

            poäng = 0 #initierar poäng variabeln

            for plats in self.profil.plats: #går igenom användarens önskade platser 
                if plats in jobb.plats: #om jobb matchar användarens önskade plats
                    poäng += 2 
                    break

            for kompetens in self.profil.kompetenser: #går igenom användarens önskade kompetenser
                if kompetens in jobb.kompetenser: # om jobb matchar användarens önskade kompetens
                    poäng += 1

            if poäng == 0: #om ingen matchning hittas på plats eller kompetens,
                poäng = 1 # men jobbet matchar användarens önskade jobb, ge 1 poäng.

            matchningar.append((jobb, poäng)) # tuppel som innehåller jobbannonsen och poängen läggs till i listan matchningar.

        matchningar.sort( #Sortera jobben efter deras poäng, från högst till lägst
            key=lambda x: x[1], # sortera efter poängen (x[1]) i matchningar. Dvs index 1 i tuplen (jobb, poäng) som vi skapade tidigare.
            reverse=True #omvänd logik, så att högsta poängen kommer först i listan.
        )

        return matchningar



## 5. Hämta jobb från JobTech API

Hämtar jobbannonser från JobTech Dev API,
skapar objekt och sparar undan rådata till fil

In [ ]:
# ========================================
#          API-FUNKTION
# ========================================

def hämta_jobb_från_api(profil): #parametern låter funktionen komma åt profil.jobb profil.plats och profil.kompetenser.
    # Hämtar jobbannonser från API:t baserat på användarens profil.

    if not profil.jobb:
        # Avbryter om användaren inte har angett något önskat jobb.
        print("Du måste ställa in din profil först.")
        return []

    # Anger adressen till API:ets sökfunktion.
    url = "https://jobsearch.api.jobtechdev.se/search"

    # Lagrar de bearbetade jobbannonserna som Jobbannons-objekt.
    jobbannonser = [] # används av matcharen för att jämföra med användarens profil.

    # Lagrar den ursprungliga datan som hämtas från API:t.
    alla_data = [] # rådata som hämtas från API:t sparas i denna lista för senare användning eller lagring.

    try:
        # Söker efter jobb för varje önskat jobb i användarens profil.
        for sökord in profil.jobb:
            print(f"\nSöker efter: {sökord}")

            # Skickar en GET-förfrågan till API:t med sökord och max antal träffar.
            response = requests.get(
                url,
                params={"q": sökord, "limit": 10},
                timeout=10
            )

            # Visar API:ets HTTP-statuskod för att se om anropet lyckades.
            print("Statuskod:", response.status_code)

            if response.status_code != 200:
                # Hoppar över sökningen om API:t inte svarade med statuskod 200.
                print("Kunde inte hämta jobbannonser.")
                continue

            # Omvandlar API:ets JSON-svar till Python-data.
            data = response.json()

            # Sparar den ursprungliga API-datan för senare användning eller lagring.
            alla_data.append(data)

            # Hämtar listan med jobbträffar från API-svaret.
            träffar = data.get("hits", []) #om programmet sakanr hits så krashar det inte pga reservvärdet -> []

            # Visar hur många jobbannonser som hittades.
            print("Antal träffar:", len(träffar))

            # Bearbetar varje jobbannons som API:t hittade.
            for jobb in träffar:

                # Hämtar jobbets titel och normaliserar texten.
                titel = normalisera(jobb.get("headline") or "")

                # Hämtar information om arbetsplatsens adress.
                adress = jobb.get("workplace_address") or {}

                # Hämtar kommunen där jobbet finns och normaliserar texten.
                plats = normalisera(
                    adress.get("municipality") or ""
                )

                # Hämtar webbadressen till jobbannonsen.
                länk = jobb.get("webpage_url") or ""

                # Hämtar informationen om jobbannonsens beskrivning.
                beskrivning = jobb.get("description") or {}

                # Hämtar själva texten från beskrivningen och normaliserar den.
                beskrivning = normalisera(
                    beskrivning.get("text") or "" 
                )

                # Lagrar de kompetenser från profilen som hittas i jobbets beskrivning.
                matchade_kompetenser = []

                # Går igenom alla kompetenser som användaren har angett.
                for kompetens in profil.kompetenser:

                    # Kontrollerar om kompetensen finns i jobbets beskrivning.
                    if kompetens in beskrivning:

                        # Sparar kompetensen om den hittades i beskrivningen.
                        matchade_kompetenser.append(kompetens)

                # Skapar ett Jobbannons-objekt och lägger det i resultatlistan.
                jobbannonser.append(
                    Jobbannons(
                        titel,
                        plats,
                        matchade_kompetenser,
                        länk
                    )
                )

        try: #try igen eftersom att vi gör något annat som kan gå fel.
            # Öppnar eller skapar en JSON-fil för att spara API-datan.
            with open("jobbdata.json", "w", encoding="utf-8") as f:

                # Sparar den hämtade API-datan som JSON i filen.
                json.dump( #sparar rå datan
                    alla_data,
                    f,
                    ensure_ascii=False, #behåller svenska tecken
                    indent=4 # json formatet.
                )

        except OSError as fel:
            # Hanterar problem som uppstår när filen ska skapas eller skrivas.
            print("\nKunde inte spara jobbdata.")
            print(fel)

        # Returnerar alla bearbetade jobbannonser.
        return jobbannonser

    except requests.exceptions.RequestException as fel:
        # Hanterar nätverksfel som uppstår när API:t kontaktas.
        print("\nEtt fel uppstod när API:t kontaktades.")
        print(fel)
        return []

    except json.JSONDecodeError:
        # Hanterar situationer där API-svaret inte kan tolkas som giltig JSON.
        print("\nKunde inte läsa svaret från API:t.")
        return []


## 6. Spara och visa historik

Hanterar sparande och visning av
tidigare sökningar och matchningar.

In [ ]:
# Sparar en genomförd sökning och dess matchningar i historiken.
def spara_historik(profil, matchningar):
    try:
        # Läser in tidigare sparad historik från JSON-filen.
        with open(
            "historik.json",
            "r",
            encoding="utf-8"
        ) as f:
            historik = json.load(f)

    # Startar med en tom historik om filen saknas eller innehåller felaktig JSON.
    except (FileNotFoundError, json.JSONDecodeError):
        historik = []

    # Skapar en post som innehåller datum, profil och matchningar för den aktuella sökningen.
    ny_sökning = {
        "datum": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "profil": {
            "jobb": profil.jobb,
            "plats": profil.plats,
            "kompetenser": profil.kompetenser
        },
        "matchningar": []
    }

    # Lägger till information om varje matchat jobb i den nya sökningen.
    for jobb, poäng in matchningar:
        ny_sökning["matchningar"].append({
            "jobb": jobb.jobb,
            "plats": jobb.plats,
            "poäng": poäng,
            "kompetenser": jobb.kompetenser,
            "länk": jobb.länk
        })

    # Lägger den nya sökningen sist i historiken.
    historik.append(ny_sökning)

    try:
        # Sparar den uppdaterade historiken i JSON-filen.
        with open(
            "historik.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                historik,
                f,
                ensure_ascii=False,
                indent=4
            )

    # Hanterar problem som uppstår när historiken ska sparas.
    except OSError as fel:
        print("\nKunde inte spara historiken.")
        print(fel)
        return

    print("\nSökningen har sparats i historiken.")


# Läser och visar tidigare sökningar från historiken.
def visa_historik():
    try:
        # Läser in historiken från JSON-filen.
        with open(
            "historik.json",
            "r",
            encoding="utf-8"
        ) as f:
            historik = json.load(f)

    # Hanterar om historikfilen saknas.
    except FileNotFoundError:
        print("\nDet finns ingen historik ännu.")
        return

    # Hanterar om historikfilen innehåller felaktig JSON.
    except json.JSONDecodeError:
        print("\nHistorikfilen kunde inte läsas.")
        return

    # Avslutar funktionen om historiken är tom.
    if not historik:
        print("\nDet finns ingen historik ännu.")
        return

    print("\n" + "=" * 40)
    print("              HISTORIK")
    print("=" * 40)

    # Går igenom varje tidigare sökning och ger den ett löpande nummer.
    for nummer, sökning in enumerate(historik, 1):
        print(f"\nSökning {nummer}")
        print(f"Datum: {sökning['datum']}")
        print(f"Jobb: {', '.join(sökning['profil']['jobb'])}")
        print(f"Plats: {', '.join(sökning['profil']['plats'])}")
        print(
            f"Kompetenser: "
            f"{', '.join(sökning['profil']['kompetenser'])}"
        )

        print("\nMatchningar:")

        # Går igenom och visar alla jobbmatchningar från den aktuella sökningen.
        for jobb in sökning["matchningar"]:
            print(f"  - {jobb['jobb']}")
            print(f"    Plats: {jobb['plats']}")
            print(f"    Poäng: {jobb['poäng']}")
            print(
                "    Matchande kompetenser: "
                + visa_kompetenser(jobb["kompetenser"])
            )
            print(f"    Länk: {jobb['länk']}")

        print("-" * 40)


## 7. Visa matchningar

Hämtar jobb från API, matchar dem mot profil,
sparar resultatet i historiken och skriver ut allt.

In [ ]:
#========================================
#       VISA MATCHNINGAR
#========================================

def visa_matchningar(profil):                          # Skapar funktionen som hämtar och visar matchningar.
    print("\nHämtar aktuella jobbannonser...")          # Informerar att jobbannonser hämtas.

    jobb_fran_api = hämta_jobb_från_api(profil)         # Hämtar jobbannonser från API:t.

    if not jobb_fran_api:                               # Kontrollerar om några jobbannonser hämtades.
        print("Inga jobbannonser kunde hämtas.")        # Informerar om att inga annonser hämtades.
        return                                          # Avslutar funktionen.

    matchningar = Matchare(                             # Skapar en Matchare för profilen och annonserna.
        profil,                                         # Skickar in användarens profil.
        jobb_fran_api                                   # Skickar in jobbannonserna.
    ).match()                                           # Kör matchningsfunktionen.

    if not matchningar:                                 # Kontrollerar om några jobb matchade profilen.
        print("\nInga jobb matchade din profil.")       # Informerar om att ingen annons matchade.
        return                                          # Avslutar funktionen.

    spara_historik(profil, matchningar)                 # Sparar matchningarna i historiken.

    print("\n" + "=" * 40)                              # Skriver ut en avgränsare.
    print("           DINA MATCHNINGAR")                # Skriver ut rubriken för matchningsresultatet.
    print("=" * 40)                                     # Skriver ut en avgränsare.

    for jobb, poäng in matchningar:                     # Går igenom jobb och deras matchningspoäng.
        print(f"\n- {jobb.jobb}")                       # Visar jobbets titel.
        print(f"  Plats: {jobb.plats}")                # Visar jobbets plats.
        print(f"  Poäng: {poäng}")                      # Visar matchningspoängen.
        print(                                          # Skriver ut matchande kompetenser.
            "  Matchande kompetenser: "                 # Skriver ut rubriken för kompetenserna.
            + visa_kompetenser(jobb.kompetenser)        # Visar de matchande kompetenserna.
        )                                               # Avslutar utskriften.
        print(f"  Länk: {jobb.länk}")                   # Visar länken till jobbet.


## 8. Användargränssnitt

Skapar en profil-instans och hanterar
programmets huvudmeny i terminalen.

In [ ]:
#========================================
#       ANVÄNDARGRÄNSSNITT
#========================================

profil = Profil()                                      # Skapar programmets profilobjekt.


def user_ui():                                         # Skapar programmets huvudmeny.
    while True:                                        # Kör menyn tills användaren avslutar.
        print("\n")                                    # Skapar en tom rad för bättre läsbarhet.
        print("=" * 40)                                # Skriver ut en avgränsare.
        print("       VÄLKOMMEN TILL AI-JOBBANALYS")  # Visar programmets titel.
        print("=" * 40)                                # Skriver ut en avgränsare.
        print("1. Ställ in profil")                    # Visar menyvalet för profil.
        print("2. Visa matchningar")                   # Visar menyvalet för matchningar.
        print("3. Historik")                           # Visar menyvalet för historik.
        print("4. Avsluta")                            # Visar menyvalet för att avsluta.
        print("=" * 40)                                # Skriver ut en avgränsare.

        try:                                           # Försöker köra kod som kan ge ett fel.
            svar = input("Välj ett alternativ 1-4: ")  # Läser användarens menyval.

        except (KeyboardInterrupt, EOFError):           # Fångar avbruten eller avslutad användarinmatning.
            print("\nProgrammet avslutas.")             # Informerar att programmet avslutas.
            break                                      # Avslutar menyn.

        if svar == "1":                                # Kontrollerar om användaren vill ställa in profilen.
            profil.input_jobb()                        # Läser in önskat jobb.
            profil.input_plats()                       # Läser in önskad plats.
            profil.input_kompetenser()                 # Läser in kompetenser.

            print("\nProfil ändrad!")                  # Bekräftar att profilen ändrats.
            print(f"\nJobb: {profil.jobb}")             # Visar valda jobb.
            print(f"Plats/Ort: {profil.plats}")        # Visar valda platser.
            print(f"Kompetenser: {profil.kompetenser}") # Visar valda kompetenser.

        elif svar == "2":                              # Kontrollerar om användaren vill se matchningar.
            visa_matchningar(profil)                   # Hämtar och visar matchande jobb.

        elif svar == "3":                              # Kontrollerar om användaren vill se historiken.
            visa_historik()                            # Visar tidigare sökningar.

        elif svar == "4":                              # Kontrollerar om användaren vill avsluta.
            print("\nTack för denna gång!")            # Visar ett avslutningsmeddelande.
            break                                      # Avslutar menyn.

        else:                                          # Hanterar ett menyval som inte är giltigt.
            print("\nFelaktigt val.")                  # Informerar om att menyvalet är fel.
            print("Välj en siffra mellan 1 och 4.")    # Ber användaren välja ett giltigt alternativ.


## 9. Starta programmet

startar Programmet
1

In [ ]:
user_ui() # Startar programmets användargränssnitt.3



## 10. Utvecklingsplan

1. Profil: Jobb, Plats, Kompetenser  
2. API: Hämta aktuella jobbannonser  
3. Matchning: Matcha jobb, plats, kompetenser  
4. Resultat: Visa jobb och matchningspoäng  
5. Förbättringar: Validering, Historik, Bättre UI  
6. Testning: Testa och färdigställ programmet